In [1]:
ocr_input_file = "/home/iceberg/data/OCR_INPUT.json"
ocr_input_file

'/home/iceberg/data/OCR_INPUT.json'

In [10]:
import json

In [11]:
with open(ocr_input_file, 'r') as file:
    data = json.load(file)

print(data)

{'OCR': [{'Staging': {'Database_Name': 'ocr_staging_db', 'Source': {'File': [{'Connector': 'CSV', 'Location': '/home/iceberg/data/taxi_zone_lookup.csv', 'Header': 'True', 'Target_Iceberg_Table': 'taxi_zone_stg', 'Sttm_Identifier': 'OCR_STG_TAXI_ZONE'}, {'Connector': 'Parquet', 'Location': '/home/iceberg/data/yellow_tripdata_2025-01.parquet', 'Target_Iceberg_Table': 'yellow_taxi_stg', 'Sttm_Identifier': 'OCR_STG_Y_TAXI'}], 'Database': [{'Connector': 'Postgres', 'Host': '0.0.0.0', 'Port': '5432', 'DB': 'spark_demo', 'User': 'admin', 'Password': 'admin', 'Source_table': 'customers', 'Target_Iceberg_Table': 'customers_stg', 'Sttm_Identifier': 'OCR_STG_CUST'}]}}}, {'Bronze': {'Database_Name': 'ocr_bronze_db', 'Properties': [{'Source_Table': 'taxi_zone_stg', 'Target_Table': 'taxi_zone_bronze', 'Sttm_Identifier': 'OCR_BRZ_CUST'}, {'Source_Table': 'yellow_taxi_stg', 'Target_Table': 'yellow_taxi_bronze', 'Sttm_Identifier': 'OCR_BRZ_Y_TAXI'}, {'Source_Table': 'customers_stg', 'Target_Table': 'cu

In [15]:
data["OCR"][0]

{'Staging': {'Database_Name': 'ocr_staging_db',
  'Source': {'File': [{'Connector': 'CSV',
     'Location': '/home/iceberg/data/taxi_zone_lookup.csv',
     'Header': 'True',
     'Target_Iceberg_Table': 'taxi_zone_stg',
     'Sttm_Identifier': 'OCR_STG_TAXI_ZONE'},
    {'Connector': 'Parquet',
     'Location': '/home/iceberg/data/yellow_tripdata_2025-01.parquet',
     'Target_Iceberg_Table': 'yellow_taxi_stg',
     'Sttm_Identifier': 'OCR_STG_Y_TAXI'}],
   'Database': [{'Connector': 'Postgres',
     'Host': '0.0.0.0',
     'Port': '5432',
     'DB': 'spark_demo',
     'User': 'admin',
     'Password': 'admin',
     'Source_table': 'customers',
     'Target_Iceberg_Table': 'customers_stg',
     'Sttm_Identifier': 'OCR_STG_CUST'}]}}}

In [16]:
print(data.keys())

dict_keys(['OCR'])


In [55]:
# Wrapper Class
import json
from typing import List, Dict, Any, Optional

class AppConfig:
    def __init__(self, config_file: str):
        with open(config_file, "r") as f:
            self.config = json.load(f)

        # Preprocess into structured dict
        self.data = {}
        for app_name, layers in self.config.items():
            self.data[app_name] = {}
            for layer_obj in layers:  # each is {"Staging": {...}}, {"Bronze": {...}}, ...
                for layer_name, details in layer_obj.items():
                    self.data[app_name][layer_name] = details

    def get_applications(self) -> str:
        """Return name of application."""
        return list(self.data.keys())[0]

    def get_layers(self, app: str) -> List[str]:
        """Return list of layers (Staging, Bronze, Silver, Gold)."""
        return list(self.data.get(app, {}).keys())

    def get_database_name(self, app: str, layer: str) -> Optional[str]:
        """Return database name for given application + layer."""
        return self.data.get(app, {}).get(layer, {}).get("Database_Name")

    # ---------- Staging Specific ----------
    def get_files(self, app: str) -> List[Dict[str, Any]]:
        """Return list of file sources (only for Staging)."""
        return self.data.get(app, {}).get("Staging", {}).get("Source", {}).get("File", [])

    def get_databases(self, app: str) -> List[Dict[str, Any]]:
        """Return list of database sources (only for Staging)."""
        return self.data.get(app, {}).get("Staging", {}).get("Source", {}).get("Database", [])

    # ---------- Bronze/Silver/Gold ----------
    def get_properties(self, app: str, layer: str) -> List[Dict[str, Any]]:
        """Return transformation properties for Bronze/Silver/Gold layers."""
        return self.data.get(app, {}).get(layer, {}).get("Properties", [])

    # ---------- Search / Utility ----------
    def search_by_source_table(self, table: str) -> List[Dict[str, str]]:
        """Find where a given source table is used (across all apps/layers)."""
        results = []
        for app, layers in self.data.items():
            for layer, details in layers.items():
                if layer == "Staging":
                    # Staging DB sources
                    for db in details.get("Source", {}).get("Database", []):
                        if db.get("Source_table") == table:
                            results.append({"App": app, "Layer": layer, "Target": db.get("Target_Iceberg_Table")})
                else:
                    # Bronze/Silver/Gold Properties
                    for prop in details.get("Properties", []):
                        if prop.get("Source_Table") == table:
                            results.append({"App": app, "Layer": layer, "Target": prop.get("Target_Table")})
        return results

    # ---------- Lineage Tracing ----------
    def trace_lineage(self, app: str, start_table: str) -> List[str]:
        """
        Trace table lineage across layers (Staging → Bronze → Silver → Gold).
        """
        lineage = [start_table]
        current = start_table

        for layer in ["Bronze", "Silver", "Gold"]:
            props = self.get_properties(app, layer)
            for p in props:
                if p.get("Source_Table") == current:
                    target = p.get("Target_Table")
                    lineage.append(target)
                    current = target
                    break  # move to next layer

        return lineage

In [56]:
config = AppConfig(ocr_input_file)

In [57]:
print("Applications:", config.get_applications())
print("OCR Layers:", config.get_layers("OCR"))

Applications: OCR
OCR Layers: ['Staging', 'Bronze', 'Silver', 'Gold']


In [45]:
print("\nStaging DB:", config.get_database_name("OCR", "Staging"))
print("Bronze DB:", config.get_database_name("OCR", "Bronze"))
print("Silver DB:", config.get_database_name("OCR", "Silver"))
print("Gold DB:", config.get_database_name("OCR", "Gold"))


Staging DB: ocr_staging_db
Bronze DB: ocr_bronze_db
Silver DB: ocr_silver_db
Gold DB: ocr_gold_db


In [46]:
print("\nStaging Files:")
for f in config.get_files("OCR"):
    print("  -", f["Connector"], "->", f["Location"], "Target ->", f["Target_Iceberg_Table"], "STTM -> ", f["Sttm_Identifier"])


Staging Files:
  - CSV -> /home/iceberg/data/taxi_zone_lookup.csv Target -> taxi_zone_stg STTM ->  OCR_STG_TAXI_ZONE
  - Parquet -> /home/iceberg/data/yellow_tripdata_2025-01.parquet Target -> yellow_taxi_stg STTM ->  OCR_STG_Y_TAXI


In [47]:
print("\nStaging Databases:")
for f in config.get_databases("OCR"):
    print("  -", f["Connector"], "->", "Host = ", f["Host"], "Target ->", f["Target_Iceberg_Table"], "STTM -> ", f["Sttm_Identifier"])


Staging Databases:
  - Postgres -> Host =  0.0.0.0 Target -> customers_stg STTM ->  OCR_STG_CUST


In [49]:
print("\nBronze Properties:")
for p in config.get_properties("OCR", "Bronze"):
    print("  -", p["Source_Table"], "->", p["Target_Table"], "\t STTM = ", p["Sttm_Identifier"])


Bronze Properties:
  - taxi_zone_stg -> taxi_zone_bronze 	 STTM =  OCR_BRZ_CUST
  - yellow_taxi_stg -> yellow_taxi_bronze 	 STTM =  OCR_BRZ_Y_TAXI
  - customers_stg -> customers_bronze 	 STTM =  OCR_BRZ_CUST


In [50]:
print("\nSilver Properties:")
for p in config.get_properties("OCR", "Silver"):
    print("  -", p["Source_Table"], "->", p["Target_Table"], "\t STTM = ", p["Sttm_Identifier"])


Silver Properties:
  - taxi_zone_bronze -> taxi_zone_silver 	 STTM =  OCR_SLR_CUST
  - yellow_taxi_bronze -> yellow_taxi_silver 	 STTM =  OCR_SLR_Y_TAXI
  - customers_bronze -> customers_silver 	 STTM =  OCR_SLR_CUST


In [54]:
print("\nSearch 'customers_stg':")
print(config.search_by_source_table("customers_stg"))


Search 'customers_stg':
[{'App': 'OCR', 'Layer': 'Bronze', 'Target': 'customers_bronze'}]


In [58]:
print("\nLineage for 'customers_stg':")
print(" → ".join(config.trace_lineage("OCR", "customers_stg")))


Lineage for 'customers_stg':
customers_stg → customers_bronze → customers_silver → customers_gold


In [18]:
with open(ocr_input_file, 'r') as file:
    config = json.load(file)

In [19]:
data = {}

In [21]:
for app_name, layers in config.items():
    print('App => ', app_name)
    data[app_name] = {}
    for layer_obj in layers:  # each is {"Staging": {...}}, {"Bronze": {...}}, ...
        for layer_name, details in layer_obj.items():
            print('Layer => ', layer_name)
            data[app_name][layer_name] = details
            print('App Layer Details => ', data[app_name][layer_name])

App =>  OCR
Layer =>  Staging
App Layer Details =>  {'Database_Name': 'ocr_staging_db', 'Source': {'File': [{'Connector': 'CSV', 'Location': '/home/iceberg/data/taxi_zone_lookup.csv', 'Header': 'True', 'Target_Iceberg_Table': 'taxi_zone_stg', 'Sttm_Identifier': 'OCR_STG_TAXI_ZONE'}, {'Connector': 'Parquet', 'Location': '/home/iceberg/data/yellow_tripdata_2025-01.parquet', 'Target_Iceberg_Table': 'yellow_taxi_stg', 'Sttm_Identifier': 'OCR_STG_Y_TAXI'}], 'Database': [{'Connector': 'Postgres', 'Host': '0.0.0.0', 'Port': '5432', 'DB': 'spark_demo', 'User': 'admin', 'Password': 'admin', 'Source_table': 'customers', 'Target_Iceberg_Table': 'customers_stg', 'Sttm_Identifier': 'OCR_STG_CUST'}]}}
Layer =>  Bronze
App Layer Details =>  {'Database_Name': 'ocr_bronze_db', 'Properties': [{'Source_Table': 'taxi_zone_stg', 'Target_Table': 'taxi_zone_bronze', 'Sttm_Identifier': 'OCR_BRZ_CUST'}, {'Source_Table': 'yellow_taxi_stg', 'Target_Table': 'yellow_taxi_bronze', 'Sttm_Identifier': 'OCR_BRZ_Y_TAXI

In [24]:
list(data.keys())[0]

'OCR'

In [26]:
list(data.get('OCR', {}).keys())

['Staging', 'Bronze', 'Silver', 'Gold']